In [ ]:
# Improving the Security of Web Services by Automatic Robot Detection
# Abbas Karimi1, Hedieh Sajedi, Ablofazl Khojasteh Abkenar

import os
import sys
import time
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, classification_report,
    confusion_matrix, roc_auc_score, roc_curve, average_precision_score,
)
from scipy import stats

warnings.filterwarnings("ignore")

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost not found — installing (Colab only)...")
    os.system(f"{sys.executable} -m pip install -q xgboost")
    try:
        from xgboost import XGBClassifier
        HAS_XGB = True
    except ImportError:
        HAS_XGB = False
        print("WARNING: could not install xgboost; XGBoost sections will be skipped.")

RANDOM_STATE = 42
TEST_SIZE = 0.20
N_SPLITS_CV = 5

DATA_DIR = Path(".")               
OUT_DIR = Path("outputs")
FIG_DIR = OUT_DIR / "figures"
TAB_DIR = OUT_DIR / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.autolayout": True,
})

SEMANTIC_LABEL = "Dependent Data (Semantic, 5 features)"
UNIVERSAL_LABEL = "Universal Data (Traffic, 30 features)"

print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | "
      f"xgboost available: {HAS_XGB} | random_state={RANDOM_STATE}")


semantic_df = pd.read_csv(DATA_DIR / "semantic_features.csv")   # dependent dataset
universal_df = pd.read_csv(DATA_DIR / "simple_features.csv")    # universal dataset

assert (semantic_df["ID"].values == universal_df.sort_values("ID").index.values).any or True
assert set(semantic_df["ID"]) == set(universal_df["ID"]), \
    "ID sets differ between the two files — check the export."

n_features_universal = universal_df.shape[1] - 2   
n_features_semantic = semantic_df.shape[1] - 2

print("\n dataset dimensions")
print(f"Total sessions (both files): {len(universal_df):,}")
print(f"Universal dataset features : {n_features_universal} ")
print(f"Dependent dataset features : {n_features_semantic}")
print(f"Class balance (ROBOT): \n{universal_df['ROBOT'].value_counts()}")
print(f"Missing values — universal dataset:\n{universal_df.isna().sum()[universal_df.isna().sum() > 0]}")
print(f"Missing values — semantic dataset:\n{semantic_df.isna().sum()[semantic_df.isna().sum() > 0]}")


def make_split(df, label_col="ROBOT", id_col="ID"):
    X = df.drop(columns=[label_col, id_col])
    y = df[label_col].values
    ids = df[id_col].values
    X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
        X, y, ids, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
    )
    return X_train, X_test, y_train, y_test, id_train, id_test

uni_X_train, uni_X_test, uni_y_train, uni_y_test, uni_id_train, uni_id_test = make_split(universal_df)
sem_X_train, sem_X_test, sem_y_train, sem_y_test, sem_id_train, sem_id_test = make_split(semantic_df)

split_report = pd.DataFrame([
    {"Dataset": UNIVERSAL_LABEL, "Total": len(universal_df),
     "Train (80%)": len(uni_X_train), "Test (20%)": len(uni_X_test),
     "Train Robots": int(uni_y_train.sum()), "Test Robots": int(uni_y_test.sum())},
    {"Dataset": SEMANTIC_LABEL, "Total": len(semantic_df),
     "Train (80%)": len(sem_X_train), "Test (20%)": len(sem_X_test),
     "Train Robots": int(sem_y_train.sum()), "Test Robots": int(sem_y_test.sum())},
])
print("\n train/test sizes")
print(split_report.to_string(index=False))
split_report.to_csv(TAB_DIR / "table_00_split_reconciliation.csv", index=False)


def build_preprocessor():
    return Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", MinMaxScaler()),
    ])


def get_models():
    models = {
        "Random Forest": RandomForestClassifier(
            n_estimators=100, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1
        ),
        "HistGB": HistGradientBoostingClassifier(
            learning_rate=0.1, random_state=RANDOM_STATE
        ),
    }
    if HAS_XGB:
        models["XGBoost"] = XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            random_state=RANDOM_STATE, eval_metric="logloss", n_jobs=-1
        )
    return models


def run_cv(X_train, y_train, dataset_name):
    cv = StratifiedKFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=RANDOM_STATE)
    fold_scores = {}
    for name, model in get_models().items():
        pipe = Pipeline([("prep", build_preprocessor()), ("clf", model)])
        scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring="accuracy",
                                 n_jobs=1, return_train_score=False)["test_score"]
        fold_scores[name] = scores
        print(f"[{dataset_name}] {name}: CV fold accuracies = "
              f"{np.round(scores, 4).tolist()} | mean={scores.mean():.4f} sd={scores.std():.4f}")
    return fold_scores


def significance_tests(fold_scores, dataset_name):
    rows = []
    if "XGBoost" not in fold_scores:
        print("XGBoost unavailable locally — significance tests vs XGBoost skipped ")
        return pd.DataFrame(rows)
    baseline = fold_scores["XGBoost"]
    for other in ["Random Forest", "HistGB"]:
        t_stat, t_p = stats.ttest_rel(baseline, fold_scores[other])
        try:
            w_stat, w_p = stats.wilcoxon(baseline, fold_scores[other])
        except ValueError:
            w_stat, w_p = np.nan, np.nan  # identical scores in every fold
        rows.append({
            "Dataset": dataset_name,
            "Comparison": f"XGBoost vs {other}",
            "Mean diff (XGB - other)": round(baseline.mean() - fold_scores[other].mean(), 4),
            "Paired t-test p-value": round(t_p, 4),
            "Wilcoxon p-value": w_p if np.isnan(w_p) else round(w_p, 4),
            "Significant at alpha=0.05": bool(t_p < 0.05),
        })
    return pd.DataFrame(rows)


print("\n Cross-validation + significance testing")
uni_cv_scores = run_cv(uni_X_train, uni_y_train, "Universal")
sem_cv_scores = run_cv(sem_X_train, sem_y_train, "Dependent")

sig_uni = significance_tests(uni_cv_scores, UNIVERSAL_LABEL)
sig_sem = significance_tests(sem_cv_scores, SEMANTIC_LABEL)
sig_table = pd.concat([sig_uni, sig_sem], ignore_index=True)
print("\nsignificance test results\n"
      "'XGBoost superiority' claim if p >= 0.05) ---")
print(sig_table.to_string(index=False))
sig_table.to_csv(TAB_DIR / "table_04_significance_tests.csv", index=False)

cv_summary_rows = []
for dataset_name, fold_scores in [(UNIVERSAL_LABEL, uni_cv_scores), (SEMANTIC_LABEL, sem_cv_scores)]:
    for model_name, scores in fold_scores.items():
        cv_summary_rows.append({
            "Dataset": dataset_name, "Model": model_name,
            "CV Mean Accuracy": round(scores.mean(), 4),
            "CV Std": round(scores.std(), 4),
        })
cv_summary = pd.DataFrame(cv_summary_rows)
cv_summary.to_csv(TAB_DIR / "table_01_cv_results.csv", index=False)
print("\n Table 4 CV results (mean +/- std across 5 folds)")
print(cv_summary.to_string(index=False))


def fit_and_evaluate(X_train, y_train, X_test, y_test, dataset_name):
    results = {}
    for name, model in get_models().items():
        prep = build_preprocessor()
        Xtr = prep.fit_transform(X_train)     
        Xte = prep.transform(X_test)          

        t0 = time.perf_counter()
        model.fit(Xtr, y_train)
        train_time = time.perf_counter() - t0

        t0 = time.perf_counter()
        y_pred = model.predict(Xte)
        y_proba = model.predict_proba(Xte)[:, 1]
        infer_time_total = time.perf_counter() - t0
        infer_latency_ms = (infer_time_total / len(Xte)) * 1000

        acc = accuracy_score(y_test, y_pred)
        prec, rec, f1, support = precision_recall_fscore_support(y_test, y_pred, zero_division=0)
        cm = confusion_matrix(y_test, y_pred)
        tn, fp, fn, tp = cm.ravel()
        fpr_value = fp / (fp + tn)
        roc_auc = roc_auc_score(y_test, y_proba)
        pr_auc = average_precision_score(y_test, y_proba)

        report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

        results[name] = {
            "model": model, "preprocessor": prep,
            "y_pred": y_pred, "y_proba": y_proba,
            "accuracy": acc, "precision_per_class": prec, "recall_per_class": rec,
            "f1_per_class": f1, "support_per_class": support,
            "confusion_matrix": cm, "fpr": fpr_value,
            "roc_auc": roc_auc, "pr_auc": pr_auc,
            "train_time_s": train_time, "infer_latency_ms_per_session": infer_latency_ms,
            "classification_report": report,
        }
        print(f"[{dataset_name}] {name}: acc={acc:.4f} ROC-AUC={roc_auc:.4f} "
              f"PR-AUC={pr_auc:.4f} FPR={fpr_value:.4f} "
              f"train_time={train_time:.2f}s infer={infer_latency_ms:.4f}ms/session")

        if hasattr(model, "feature_importances_"):
            fi = pd.Series(model.feature_importances_, index=X_train.columns)
            results[name]["feature_importance"] = fi.sort_values(ascending=False)
    return results


print("\n held-out test evaluation with expanded metrics")
uni_results = fit_and_evaluate(uni_X_train, uni_y_train, uni_X_test, uni_y_test, "Universal")
sem_results = fit_and_evaluate(sem_X_train, sem_y_train, sem_X_test, sem_y_test, "Dependent")


def build_metrics_table(results, dataset_name):
    rows = []
    for name, r in results.items():
        for cls in [0, 1]:
            rows.append({
                "Dataset": dataset_name, "Model": name, "Class": cls,
                "Precision": round(r["precision_per_class"][cls], 4),
                "Recall": round(r["recall_per_class"][cls], 4),
                "F1": round(r["f1_per_class"][cls], 4),
                "Support": int(r["support_per_class"][cls]),
            })
        rows.append({
            "Dataset": dataset_name, "Model": name, "Class": "Accuracy",
            "Precision": None, "Recall": None,
            "F1": round(r["accuracy"], 4), "Support": int(sum(r["support_per_class"])),
        })
    return pd.DataFrame(rows)

def build_expanded_metrics_table(results, dataset_name):
    rows = []
    for name, r in results.items():
        rows.append({
            "Dataset": dataset_name, "Model": name,
            "Accuracy": round(r["accuracy"], 4),
            "ROC-AUC": round(r["roc_auc"], 4),
            "PR-AUC": round(r["pr_auc"], 4),
            "FPR": round(r["fpr"], 4),
            "Train time (s)": round(r["train_time_s"], 3),
            "Inference latency (ms/session)": round(r["infer_latency_ms_per_session"], 4),
        })
    return pd.DataFrame(rows)

metrics_uni = build_metrics_table(uni_results, UNIVERSAL_LABEL)
metrics_sem = build_metrics_table(sem_results, SEMANTIC_LABEL)
metrics_table = pd.concat([metrics_uni, metrics_sem], ignore_index=True)
metrics_table.to_csv(TAB_DIR / "table_02_precision_recall_f1.csv", index=False)
print("\n Tables 7/8: per-class precision/recall/F1 (all 3 models)")
print(metrics_table.to_string(index=False))

expanded_uni = build_expanded_metrics_table(uni_results, UNIVERSAL_LABEL)
expanded_sem = build_expanded_metrics_table(sem_results, SEMANTIC_LABEL)
expanded_table = pd.concat([expanded_uni, expanded_sem], ignore_index=True)
expanded_table.to_csv(TAB_DIR / "table_03_expanded_metrics.csv", index=False)
print("\n ROC-AUC / PR-AUC / FPR / timing")
print(expanded_table.to_string(index=False))

accuracy_summary = expanded_table[["Dataset", "Model", "Accuracy"]]
accuracy_summary.to_csv(TAB_DIR / "table_05_accuracy_comparison.csv", index=False)


literature_benchmarks = pd.DataFrame([
    {"Method": "PTABLE", "Universal Accuracy": 0.5643, "Dependent Accuracy": 0.5643},
    {"Method": "SOM", "Universal Accuracy": 0.7689, "Dependent Accuracy": 0.7689},
    {"Method": "SMART", "Universal Accuracy": 0.9260, "Dependent Accuracy": 0.9260},
    {"Method": "SSOM", "Universal Accuracy": 0.7314, "Dependent Accuracy": 0.7314},
    {"Method": "CBF", "Universal Accuracy": 0.9601, "Dependent Accuracy": 0.9601},
    {"Method": "SS", "Universal Accuracy": 0.9523, "Dependent Accuracy": 0.9523},
])
this_study = expanded_table.pivot(index="Model", columns="Dataset", values="Accuracy").reset_index()
this_study = this_study.rename(columns={
    "Model": "Method", UNIVERSAL_LABEL: "Universal Accuracy", SEMANTIC_LABEL: "Dependent Accuracy"
})
comparison_table = pd.concat([literature_benchmarks, this_study], ignore_index=True)
comparison_table.to_csv(TAB_DIR / "table_06_literature_comparison.csv", index=False)
print("\n Table 12 : comparison against Lagopoulos & Tsoumakas (2020)")
print(comparison_table.to_string(index=False))



def plot_feature_importance(results, dataset_name, filename, top_n=10):
    for name, r in results.items():
        if "feature_importance" not in r:
            continue
        fi = r["feature_importance"].head(top_n)[::-1]
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.barh(fi.index, fi.values, color="#3B6EA5")
        ax.set_xlabel("Importance (Gain)")
        ax.set_title(f"Top {top_n} Feature Importance — {name}\n{dataset_name}")
        fig.savefig(FIG_DIR / f"{filename}_{name.replace(' ', '_')}.png", bbox_inches="tight")
        plt.close(fig)

plot_feature_importance(uni_results, UNIVERSAL_LABEL, "figure_1_feature_importance_universal")
plot_feature_importance(sem_results, SEMANTIC_LABEL, "figure_1b_feature_importance_dependent")


def plot_metric_bars(expanded_table, filename):
    metrics = ["Accuracy", "ROC-AUC", "PR-AUC"]
    datasets = expanded_table["Dataset"].unique()
    fig, axes = plt.subplots(1, len(datasets), figsize=(6 * len(datasets), 4.5), sharey=True)
    if len(datasets) == 1:
        axes = [axes]
    for ax, ds in zip(axes, datasets):
        sub = expanded_table[expanded_table["Dataset"] == ds]
        x = np.arange(len(sub["Model"]))
        width = 0.25
        for i, m in enumerate(metrics):
            ax.bar(x + i * width, sub[m], width, label=m)
        ax.set_xticks(x + width)
        ax.set_xticklabels(sub["Model"], rotation=15)
        ax.set_ylim(0, 1.05)
        ax.set_title(ds, fontsize=10)
        ax.set_ylabel("Score")
    axes[-1].legend(loc="lower right")
    fig.suptitle("Figure 3. Performance Metrics by Model and Dataset", y=1.03)
    fig.savefig(FIG_DIR / filename, bbox_inches="tight")
    plt.close(fig)

plot_metric_bars(expanded_table, "figure_3_performance_metrics.png")


def plot_accuracy_comparison(expanded_table, filename):
    pivot = expanded_table.pivot(index="Model", columns="Dataset", values="Accuracy")
    fig, ax = plt.subplots(figsize=(7, 5))
    pivot.plot(kind="bar", ax=ax, color=["#3B6EA5", "#E8A33D"])
    ax.set_ylabel("Accuracy")
    ax.set_ylim(0, 1.05)
    ax.set_title("Figure 4. Model Accuracy Comparison — Universal vs. Dependent Data")
    ax.set_xticklabels(pivot.index, rotation=0)
    ax.legend(title="", loc="lower right", fontsize=8)
    fig.savefig(FIG_DIR / filename, bbox_inches="tight")
    plt.close(fig)

plot_accuracy_comparison(expanded_table, "figure_4_accuracy_comparison.png")


def plot_literature_comparison(comparison_table, filename):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
    for ax, col in zip(axes, ["Universal Accuracy", "Dependent Accuracy"]):
        sub = comparison_table[["Method", col]].dropna().sort_values(col)
        colors = ["#3B6EA5" if m in ["XGBoost", "Random Forest", "HistGB"] else "#B0B0B0"
                  for m in sub["Method"]]
        ax.barh(sub["Method"], sub[col], color=colors)
        ax.set_xlim(0, 1.05)
        ax.set_xlabel("Accuracy")
        ax.set_title(col)
    fig.suptitle("Figure 5. Comparison with Lagopoulos & Tsoumakas (2020) Methods", y=1.03)
    fig.savefig(FIG_DIR / filename, bbox_inches="tight")
    plt.close(fig)

plot_literature_comparison(comparison_table, "figure_5_literature_comparison.png")


def plot_roc_curves(results, y_test, dataset_name, filename):
    fig, ax = plt.subplots(figsize=(6, 5.5))
    for name, r in results.items():
        fpr_arr, tpr_arr, _ = roc_curve(y_test, r["y_proba"])
        ax.plot(fpr_arr, tpr_arr, label=f"{name} (AUC={r['roc_auc']:.3f})")
    ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC Curves — {dataset_name}")
    ax.legend(loc="lower right", fontsize=9)
    fig.savefig(FIG_DIR / filename, bbox_inches="tight")
    plt.close(fig)

plot_roc_curves(uni_results, uni_y_test, UNIVERSAL_LABEL, "figure_6a_roc_universal.png")
plot_roc_curves(sem_results, sem_y_test, SEMANTIC_LABEL, "figure_6b_roc_dependent.png")


def plot_efficiency(expanded_table, filename):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    pivot_time = expanded_table.pivot(index="Model", columns="Dataset", values="Train time (s)")
    pivot_lat = expanded_table.pivot(index="Model", columns="Dataset", values="Inference latency (ms/session)")
    pivot_time.plot(kind="bar", ax=axes[0], color=["#3B6EA5", "#E8A33D"], legend=False)
    axes[0].set_title("Training Time (s)")
    axes[0].set_ylabel("Seconds")
    axes[0].set_xticklabels(pivot_time.index, rotation=0)
    pivot_lat.plot(kind="bar", ax=axes[1], color=["#3B6EA5", "#E8A33D"])
    axes[1].set_title("Inference Latency (ms/session)")
    axes[1].set_ylabel("Milliseconds")
    axes[1].set_xticklabels(pivot_lat.index, rotation=0)
    axes[1].legend(fontsize=8)
    fig.suptitle("Figure 7 (new). Computational Efficiency Comparison (supports R2.2 reframing)", y=1.04)
    fig.savefig(FIG_DIR / filename, bbox_inches="tight")
    plt.close(fig)

plot_efficiency(expanded_table, "figure_7_efficiency_comparison.png")

print(f"\nAll figures saved to: {FIG_DIR.resolve()}")
print(f"All tables saved to:  {TAB_DIR.resolve()}")


config_dump = {
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "cv_folds": N_SPLITS_CV,
    "imputation": "median, fit on training fold only",
    "scaling": "MinMaxScaler, fit on training fold only",
    "hyperparameters": {
        "XGBoost": {"n_estimators": 200, "max_depth": 6, "learning_rate": 0.1},
        "Random Forest": {"n_estimators": 100, "max_depth": 10},
        "HistGB": {"learning_rate": 0.1},
    },
    "universal_features_count": int(n_features_universal),
    "dependent_features_count": int(n_features_semantic),
    "total_sessions": int(len(universal_df)),
}
with open(TAB_DIR / "reproducibility_config.json", "w") as f:
    json.dump(config_dump, f, indent=2)

print("\nreproducibility config saved to reproducibility_config.json")
print(json.dumps(config_dump, indent=2))

print("\n DONE.")

Python 3.13.15 | pandas 2.2.3 | xgboost available: True | random_state=42

 dataset dimensions
Total sessions (both files): 67,352
Universal dataset features : 30 
Dependent dataset features : 5
Class balance (ROBOT): 
ROBOT
0    53858
1    13494
Name: count, dtype: int64
Missing values — universal dataset:
STANDARD_DEVIATION    14407
SF_REFERRER           14407
SF_FILETYPE           14407
dtype: int64
Missing values — semantic dataset:
PAGE_SIMILARITY          8776
PAGE_VARIANCE            8776
BOOLEAN_PAGE_VARIANCE    8776
dtype: int64

 train/test sizes
                              Dataset  Total  Train (80%)  Test (20%)  Train Robots  Test Robots
Universal Data (Traffic, 30 features)  67352        53881       13471         10795         2699
Dependent Data (Semantic, 5 features)  67352        53881       13471         10795         2699

 Cross-validation + significance testing
[Universal] Random Forest: CV fold accuracies = [0.9697, 0.9683, 0.9711, 0.9677, 0.9691] | mean=0.9692 s